In [ ]:
!pip install transformers accelerate peft trl datasets bitsandbytes -q


In [ ]:
!pip install -U bitsandbytes -q

In [ ]:
from google.colab import files

uploaded=files.upload()

Saving dataset_train.json to dataset_train (1).json
Saving dataset_val.json to dataset_val (1).json


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import torch

model_name="Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name)

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
import json
from datasets import Dataset

with open('dataset_train.json','r') as f1, open('dataset_val.json','r') as f2:
  train_file=json.load(f1)
  val_file=json.load(f2)

system_prompt = """You are a product data extraction assistant. Extract structured information from the given product description and return a valid JSON object with these fields:
- product_name: string
- brand: string or null if unknown
- category: string
- price: {"amount": number, "currency": string} or null if not mentioned
- key_specs: object with relevant specifications
- condition: "new", "used", or "refurbished"

Return ONLY the JSON object, no other text."""

def format_example(example):

  if isinstance(example["output"],str):
    assistant_content=example["output"]
  else:
    assistant_content=json.dumps(example["output"], indent=2)

  messages=[
      {'role':'system', 'content':system_prompt},
      {'role':'user', 'content':f"Extract the product data from this description:\n\n{example['input']}"},
      {'role':'assistant', 'content':assistant_content}
  ]

  prompt=tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=False
  )

  return {"text":prompt}

train_formatted=[format_example(example) for example in train_file]
train_dataset= Dataset.from_list(train_formatted)

val_formatted = [format_example(example) for example in val_file]
val_dataset = Dataset.from_list(val_formatted)

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

Training examples: 640
Validation examples: 80


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none"
)

model=get_peft_model(model,lora_config)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
from trl import SFTTrainer, SFTConfig

train_arg = SFTConfig(
    num_train_epochs=4,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    gradient_accumulation_steps=4,
    output_dir="./qwen-lora-output",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    warmup_steps=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    args=train_arg,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/640 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.586468,0.557371,0.583237,0.886212,194602.000000
2,0.463810,0.452332,0.495511,0.903111,389204.000000
3,0.429637,0.435082,0.464247,0.906753,583806.000000
4,0.428310,0.434034,0.459249,0.906916,778408.000000


TrainOutput(global_step=160, training_loss=0.6177912324666976, metrics={'train_runtime': 665.792, 'train_samples_per_second': 3.845, 'train_steps_per_second': 0.24, 'total_flos': 6738455036805120.0, 'train_loss': 0.6177912324666976, 'epoch': 4.0})

In [ ]:
model.save_pretrained("qwen-lora-adapter")
tokenizer.save_pretrained("qwen-lora-adapter")
print("Adapter saved!")

Adapter saved!


In [ ]:
!ls ./qwen-lora-output/

checkpoint-120	checkpoint-160	checkpoint-40  checkpoint-80  README.md


In [ ]:
import shutil
shutil.make_archive("qwen-lora-adapter", "zip", "qwen-lora-adapter")

from google.colab import files
files.download("qwen-lora-adapter.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>